# Sudanese Regional LLM - Kaggle QLoRA Fine-Tuning (Background Run)

This notebook fine-tunes **Qwen/Qwen2.5-7B-Instruct** on Sudanese dialectal datasets using **QLoRA** (4-bit quantization).
When executed via **Save & Run All (Commit)**, Kaggle will run this in the background for up to 12 hours even if you close your browser.

In [ ]:
# Install dependencies
!pip install -q -U torch transformers datasets trl peft bitsandbytes huggingface_hub

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Authenticate with Hugging Face via Kaggle Secrets (or environment variable)
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face")
except Exception as e:
    print(f"⚠️ Hugging Face token not found in Kaggle Secrets: {e}")

In [ ]:
# Dataset: Sudanese Dialect Knowledge Base
training_data = [
    {"instruction": "ما هي اللهجة المستخدمة في الخرطوم وما أبرز الملامح؟", "response": "اللهجة الخرطومية هي اللهجة الحضرية المركزية في السودان، وتستخدم كلمات مثل 'زول' (شخص)، 'كيفنك' (كيف حالك)، و'حبابك' (أهلاً بك)."},
    {"instruction": "ترجم بلهجة دارفور: كيف حالك يا صديقي؟", "response": "اللهجة الدارفورية: 'أبوي كيفنك والمال عافية؟' أو 'يا زول كيفك وكيف أمورك؟'"},
    {"instruction": "ما هو المثل الشعبي الشائع في كردفان للتضامن؟", "response": "في كردفان يُقال: 'الصلح خير والإيد الواحدة ما بتصفق' للحث على النفير والتعاون في حصاد الصمغ العربي والزراعة."},
    {"instruction": "ما معنى كلمة 'شنقلي' أو 'حبابك عشام' في الشمالية؟", "response": "في الشمالية والمنطقة النوبية، 'حبابك عشرة بلا كشرة' تعني الترحيب الحار والكرام الحفي بالأضياف."},
    {"instruction": "كيف يرحب أهل الشرق (البجا) بالضيف؟", "response": "في شرق السودان يُقال 'عافيات' و'حبابك' مع تقديم القهوة الجبنة كرمز للإكرام والضيافة البجاوية."}
]

# Format for Qwen Chat Template
def format_prompts(batch):
    formatted = []
    for inst, resp in zip(batch['instruction'], batch['response']):
        text = f"<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{resp}<|im_end|>"
        formatted.append(text)
    return {"text": formatted}

dataset = Dataset.from_list(training_data).map(format_prompts, batched=True)
print(f"✅ Dataset prepared: {len(dataset)} examples")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA Adapter Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./sudanese-llm-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="no",
    push_to_hub=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args
)

print("🚀 Starting QLoRA Fine-Tuning...")
trainer.train()
print("✅ Training Complete!")

In [ ]:
# Save trained LoRA adapters locally
model.save_pretrained("./sudanese-llm-lora-final")
tokenizer.save_pretrained("./sudanese-llm-lora-final")

# Push to Hugging Face Hub if token available
try:
    model.push_to_hub("goro806/sudanese-llm-lora")
    tokenizer.push_to_hub("goro806/sudanese-llm-lora")
    print("🎉 Successfully pushed fine-tuned LoRA adapters to Hugging Face Hub!")
except Exception as e:
    print(f"⚠️ Could not push to HF Hub: {e}")